# Task 4 — Forecasting Access and Usage (2025–2027)

Objective: produce baseline and event-augmented forecasts for Account Ownership (`ACC_OWNERSHIP`) and Digital Payment Usage (estimated from `ACC_MM_ACCOUNT` × `USG_ACTIVE_RATE`), with scenarios and confidence intervals.

This notebook contains:
- Data loading and preprocessing
- Trend models (linear) with confidence intervals
- Event-augmented forecasts using documented event effects
- Scenario analysis (pessimistic / base / optimistic)
- Forecast tables and visualizations saved to `reports/analysis_outputs/`

Run order: execute cells sequentially.

In [ ]:
# Imports and paths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

ROOT = Path('..')
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW = ROOT / 'data' / 'raw'
OUT = Path('reports') / 'analysis_outputs'
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

In [ ]:
# Load enriched dataset and impact links
enriched_fp = DATA_PROCESSED / 'ethiopia_fi_enriched_data.csv'
impact_fp = DATA_RAW / 'Impact_sheet.csv'
enriched = pd.read_csv(enriched_fp)
impact = pd.read_csv(impact_fp)

# Quick preview
enriched.head()
impact.head()

## Helper functions: annual series, usage proxy, and trend CI

In [ ]:
def annual_observed(enriched, indicator_code):
    df = enriched[enriched['indicator_code'] == indicator_code].copy()
    df = df[df['record_type'] == 'observation']
    df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
    df['year'] = df['observation_date'].dt.year.fillna(df['fiscal_year']).astype('Int64')
    s = df.groupby('year')['value_numeric'].mean().sort_index()
    s.index = s.index.astype(int)
    return s

def digital_usage_series(enriched):
    # proxy = account rate (%) * activity rate (%) -> percent of adults
    acc = annual_observed(enriched, 'ACC_MM_ACCOUNT')
    act = annual_observed(enriched, 'USG_ACTIVE_RATE')
    years = sorted(set(acc.index) | set(act.index))
    out = pd.Series(index=years, dtype=float)
    for y in years:
        a = acc.get(y, np.nan)
        r = act.get(y, np.nan)
        if pd.isna(a) and pd.isna(r):
            out.loc[y] = np.nan
        else:
            out.loc[y] = (0 if pd.isna(a) else a) * (0 if pd.isna(r) else r) / 100.0
    return out.sort_index()

def fit_linear_ci(series):
    import statsmodels.api as sm
    df = series.dropna().reset_index()
    df.columns = ['year', 'y']
    X = sm.add_constant(df['year'])
    model = sm.OLS(df['y'], X).fit()
    return model

def predict_with_ci(model, years):
    import statsmodels.api as sm
    Xnew = sm.add_constant(pd.Series(years))
    pred = model.get_prediction(Xnew)
    summary = pred.summary_frame(alpha=0.05)
    out = summary[['mean','mean_ci_lower','mean_ci_upper']].copy()
    out.index = list(years)
    out.index.name = 'year'
    return out

In [ ]:
# Prepare observed series
obs_acc = annual_observed(enriched, 'ACC_OWNERSHIP')
obs_usage = digital_usage_series(enriched)

print('Account Ownership (observed):')
display(obs_acc)
print('Digital Payment Usage (proxy) — ACC_MM_ACCOUNT * USG_ACTIVE_RATE:')
display(obs_usage)


## Trend model (linear) with confidence intervals

In [ ]:
forecast_years = [2025, 2026, 2027]
model_acc = fit_linear_ci(obs_acc)
pred_acc_trend = predict_with_ci(model_acc, forecast_years)
model_usage = fit_linear_ci(obs_usage)
pred_usage_trend = predict_with_ci(model_usage, forecast_years)

display(pred_acc_trend)
display(pred_usage_trend)


## Event-augmented forecasts and scenarios
We apply documented event effects (from `Impact_sheet.csv`) additively. Scenario multipliers: pessimistic=0.5, base=1.0, optimistic=1.25.

In [ ]:
# Prepare effects table
events = enriched[enriched['record_type'] == 'event'].rename(columns={'record_id':'parent_id','indicator':'event_name','indicator_code':'event_code','observation_date':'event_date'})
effects = impact.merge(events[['parent_id','event_name','event_code','event_date']], on='parent_id', how='left')
effects['impact_magnitude_num'] = pd.to_numeric(effects['impact_magnitude'], errors='coerce')
effects['impact_estimate_num'] = pd.to_numeric(effects['impact_estimate'], errors='coerce')
effects['effect_value'] = effects['impact_magnitude_num'].fillna(effects['impact_estimate_num']).fillna(0)
mask_decrease = effects['impact_direction'].astype(str).str.lower().isin(['decrease','negative'])
effects.loc[mask_decrease & effects['effect_value'].notna() & (effects['effect_value']>0),'effect_value'] *= -1
effects['event_date'] = pd.to_datetime(effects['event_date'], errors='coerce')
effects['event_year'] = effects['event_date'].dt.year

def event_effects_by_year(effects, indicator_code, multiplier=1.0):
    sub = effects[effects['related_indicator'] == indicator_code].copy()
    by_year = sub.groupby('event_year')['effect_value'].sum() * multiplier
    by_year.index = by_year.index.astype('Int64')
    return by_year

scenarios = {'pessimistic':0.5, 'base':1.0, 'optimistic':1.25}
forecasts = []
for name, mult in scenarios.items():
    eff_acc = event_effects_by_year(effects, 'ACC_OWNERSHIP', multiplier=mult)
    acc_base = pred_acc_trend['mean']
    acc_pred = acc_base.copy()
    for y in acc_pred.index:
        if y in eff_acc.index and not pd.isna(eff_acc.loc[y]):
            acc_pred.loc[y] = acc_pred.loc[y] + eff_acc.loc[y]
    acc_ci_low = pred_acc_trend['mean_ci_lower'] + (acc_pred - acc_base)
    acc_ci_high = pred_acc_trend['mean_ci_upper'] + (acc_pred - acc_base)

    eff_usage = event_effects_by_year(effects, 'ACC_MM_ACCOUNT', multiplier=mult)
    usage_base = pred_usage_trend['mean']
    usage_pred = usage_base.copy()
    # scale account effects to usage using last observed ratio where available
    for y in usage_pred.index:
        if y in eff_usage.index and not pd.isna(eff_usage.loc[y]):
            # infer last observed fraction usage/account
            acc_series = annual_observed(enriched,'ACC_MM_ACCOUNT').dropna()
            usage_series = obs_usage.dropna()
            last_frac = None
            common_years = sorted(set(acc_series.index) & set(usage_series.index))
            if common_years:
                ly = common_years[-1]
                if acc_series.get(ly, np.nan):
                    last_frac = usage_series.get(ly, 0) / acc_series.get(ly)
            if last_frac is None or pd.isna(last_frac):
                last_frac = 0.5
            usage_pred.loc[y] = usage_pred.loc[y] + eff_usage.loc[y] * last_frac
    usage_ci_low = pred_usage_trend['mean_ci_lower'] + (usage_pred - usage_base)
    usage_ci_high = pred_usage_trend['mean_ci_upper'] + (usage_pred - usage_base)

    df = pd.DataFrame({
        'year': acc_pred.index.astype(int),
        'scenario': name,
        'acc_pred_mean': acc_pred.values,
        'acc_ci_low': acc_ci_low.values,
        'acc_ci_high': acc_ci_high.values,
        'usage_pred_mean': usage_pred.values,
        'usage_ci_low': usage_ci_low.values,
        'usage_ci_high': usage_ci_high.values
    })
    forecasts.append(df)

forecasts_df = pd.concat(forecasts, ignore_index=True)
forecasts_df.to_csv(OUT / 'forecast_2025_2027_scenarios.csv', index=False)
print('Saved forecast table to', OUT / 'forecast_2025_2027_scenarios.csv')
forecasts_df.head()

In [ ]:
# Visualization: historical + trend + scenario bands for Account Ownership
plt.figure(figsize=(8,4))
plt.plot(obs_acc.index, obs_acc.values, 'o-', label='Observed')
plt.plot(pred_acc_trend.index, pred_acc_trend['mean'], '--', color='C1', label='Trend (mean)')
plt.fill_between(pred_acc_trend.index, pred_acc_trend['mean_ci_lower'], pred_acc_trend['mean_ci_upper'], color='C1', alpha=0.2, label='Trend 95% CI')
for sc in forecasts_df['scenario'].unique():
    sub = forecasts_df[forecasts_df['scenario']==sc].set_index('year')
    plt.plot(sub.index, sub['acc_pred_mean'], label=f'Scenario: {sc}')
    plt.fill_between(sub.index, sub['acc_ci_low'], sub['acc_ci_high'], alpha=0.15)
plt.title('Account Ownership — observed, trend, and scenario forecasts (2025–2027)')
plt.ylabel('Percent of adults')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'forecast_acc_2025_2027.png', dpi=150)
plt.show()

# Visualization for Digital Usage
plt.figure(figsize=(8,4))
plt.plot(obs_usage.index, obs_usage.values, 'o-', label='Observed (proxy)')
plt.plot(pred_usage_trend.index, pred_usage_trend['mean'], '--', color='C1', label='Trend (mean)')
plt.fill_between(pred_usage_trend.index, pred_usage_trend['mean_ci_lower'], pred_usage_trend['mean_ci_upper'], color='C1', alpha=0.2, label='Trend 95% CI')
for sc in forecasts_df['scenario'].unique():
    sub = forecasts_df[forecasts_df['scenario']==sc].set_index('year')
    plt.plot(sub.index, sub['usage_pred_mean'], label=f'Scenario: {sc}')
    plt.fill_between(sub.index, sub['usage_ci_low'], sub['usage_ci_high'], alpha=0.15)
plt.title('Digital Payment Usage (proxy) — observed, trend, and scenarios')
plt.ylabel('Percent of adults (proxy)')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'forecast_usage_2025_2027.png', dpi=150)
plt.show()

## Interpretation & Limitations
- Baseline trend is a linear continuation of historical means; confidence intervals reflect OLS parameter uncertainty.
- Event-augmented forecasts add documented event effects additively. This notebook uses simple scenario multipliers; effects are treated as percentage-point changes unless documented otherwise.
- Key uncertainties: measurement sparsity (few Findex points), unit heterogeneity (pp vs % vs counts), timing and persistence of event effects, and unobserved interactions.

**Files produced**: `forecast_2025_2027_scenarios.csv`, `forecast_acc_2025_2027.png`, `forecast_usage_2025_2027.png` in `reports/analysis_outputs/`.